## Setup

In [ ]:
import os, json
from dotenv import load_dotenv
from garminconnect import Garmin
from datetime import date, timedelta

load_dotenv()

EMAIL = os.getenv("GARMIN_EMAIL")
PASSWORD = os.getenv("GARMIN_PASSWORD")
# TOKEN_DIR = os.path.expanduser("~/.garminconnect")

os.environ["REQUESTS_CA_BUNDLE"] = r"C:\Certificates\gc-bundle.pem"

## Initial login 

In [ ]:
client = Garmin(EMAIL, PASSWORD)
client.login()

# Confirm it worked
print("Logged in as:", client.get_full_name())

## Test tokens - TODO

## Using the funcs

In [ ]:
# Additional imports
import pandas as pd

### Test 1 - List of Activities

In [ ]:
today = date.today().isoformat()
yesterday = (date.today() - timedelta(days=1)).isoformat()

# Get recent activities
activities = client.get_activities(0, 100, activitytype='running')

# print keys stored in an activity?
if activities:
    print("Activity keys:", activities[0].keys())

rows = []
for a in activities:
    rows.append({
        'id': a['activityId'],
        'name': a['activityName'],
        'type': a['activityType']['typeKey'],
        'start_time': a['startTimeLocal'],
        'distance': a.get('distance', 0) or 0,
        'duration': a['duration']
    })

df = pd.DataFrame(rows)
df.head()

### Test 2 - Raw Extraction

In [ ]:
# Get a single activity summary
activity_id = activities[2]['activityId']
activity = client.get_activity(activity_id)

# Inspect structure
print("Activity keys:", activity.keys())

print(json.dumps(activity, indent=2))

### Test 3 - Cleaner Extraction

#### Cycling
`22492461814`

In [ ]:
activity = client.get_activity(22492461814)

# Extract the fields worth exposing in an MCP tool
summary = activity['summaryDTO']
meta = activity['activityTypeDTO']

clean = {
    'id': activity['activityId'],
    'name': activity['activityName'],
    'type': meta['typeKey'],
    'date': summary['startTimeLocal'],
    'location': activity.get('locationName'),
    'distance_km': round(summary.get('distance', 0) / 1000, 2),
    'duration_min': round(summary.get('duration', 0) / 60, 1),
    'elevation_gain_m': summary.get('elevationGain'),
    'avg_speed_kph': round(summary.get('averageSpeed', 0) * 3.6, 1),
    'avg_hr': summary.get('averageHR'),
    'max_hr': summary.get('maxHR'),
    # Cycling-specific
    'avg_power': summary.get('averagePower'),
    'normalized_power': summary.get('normalizedPower'),
    'tss': summary.get('trainingStressScore'),
    'intensity_factor': summary.get('intensityFactor'),
    'avg_cadence': summary.get('averageBikeCadence'),
    'calories': summary.get('calories'),
    'training_effect': summary.get('trainingEffect'),
    'training_load': summary.get('activityTrainingLoad'),
}

import json
print(json.dumps(clean, indent=2))

#### Extraction func

In [ ]:
def extract_activity_summary(activity: dict) -> dict:
    summary = activity['summaryDTO']
    meta = activity['activityTypeDTO']
    sport = meta['typeKey']
    
    base = {
        'id': activity['activityId'],
        'name': activity['activityName'],
        'type': sport,
        'date': summary['startTimeLocal'],
        'location': activity.get('locationName'),
        'distance_km': round((summary.get('distance') or 0) / 1000, 2),
        'duration_min': round((summary.get('duration') or 0) / 60, 1),
        'elevation_gain_m': summary.get('elevationGain'),
        'avg_hr': summary.get('averageHR'),
        'max_hr': summary.get('maxHR'),
        'calories': summary.get('calories'),
        'training_effect': round(summary.get('trainingEffect') or 0, 1),
        'training_load': round(summary.get('activityTrainingLoad') or 0, 1),
        'training_effect_label': summary.get('trainingEffectLabel'),
        'avg_respiration': round(summary.get('avgRespirationRate') or 0, 1),
        'stamina_end': summary.get('endPotentialStamina'),
    }
    
    if sport in ('road_biking', 'cycling', 'indoor_cycling'):
        base.update({
            'avg_speed_kph': round((summary.get('averageSpeed') or 0) * 3.6, 1),
            'avg_power': summary.get('averagePower'),
            'normalized_power': summary.get('normalizedPower'),
            'tss': round(summary.get('trainingStressScore') or 0, 1),
            'intensity_factor': round(summary.get('intensityFactor') or 0, 2),
            'ftp': summary.get('functionalThresholdPower'),
            'avg_cadence': summary.get('averageBikeCadence'),
            'seated_time_min': round((summary.get('seatedTime') or 0) / 60, 1),
        })
    
    elif sport == 'running':
        avg_speed = summary.get('averageSpeed') or 0
        gap_speed = summary.get('avgGradeAdjustedSpeed') or 0
        base.update({
            'avg_pace_min_km': round(1 / (avg_speed * 0.06), 2) if avg_speed > 0 else None,
            'grade_adjusted_pace': round(1 / (gap_speed * 0.06), 2) if gap_speed > 0 else None,
            'avg_power': summary.get('averagePower'),
            'normalized_power': summary.get('normalizedPower'),
            'avg_cadence': round(summary.get('averageRunCadence') or 0),
            'ground_contact_ms': round(summary.get('groundContactTime') or 0, 1),
            'stride_length_cm': round(summary.get('strideLength') or 0, 1),
            'vertical_oscillation_cm': round(summary.get('verticalOscillation') or 0, 2),
            'vertical_ratio': round(summary.get('verticalRatio') or 0, 2),
            'rpe': summary.get('directWorkoutRpe'),
            'body_battery_change': summary.get('differenceBodyBattery'),
        })
    
    elif sport == 'lap_swimming':
        pool_length = summary.get('poolLength') or 0
        active_lengths = summary.get('numberOfActiveLengths') or 0
        base.update({
            'pool_length_m': pool_length,
            'active_lengths': active_lengths,
            'active_distance_km': round((pool_length * active_lengths) / 1000, 2),
            'moving_duration_min': round((summary.get('movingDuration') or 0) / 60, 1),
            'avg_cadence': summary.get('averageSwimCadence'),
            'avg_strokes_per_length': round(summary.get('averageStrokes') or 0, 1),
            'swolf': summary.get('averageSWOLF'),
            'total_strokes': summary.get('totalNumberOfStrokes'),
            'anaerobic_training_effect': round(summary.get('anaerobicTrainingEffect') or 0, 1),
            'rpe': summary.get('directWorkoutRpe'),
        })
    
    return base

#### Testing the func

In [ ]:
bike_activity = client.get_activity(22492461814)  
run_activity = client.get_activity(22545458432)   
swim_activity = client.get_activity(22516481045) 

# Test on all three
cycling_clean = extract_activity_summary(bike_activity)  # your cycling activity
run_clean = extract_activity_summary(run_activity)   # your run activity
swim_clean = extract_activity_summary(swim_activity)   # your swim activity

print("CYCLING:")
print(json.dumps(cycling_clean, indent=2))
print("\nRUN:")
print(json.dumps(run_clean, indent=2))
print("\nSWIM:")
print(json.dumps(swim_clean, indent=2))

### Test 4 - Additional Extraction

#### Test features

In [ ]:
# 1. Check available methods on the client
methods = [m for m in dir(client) if not m.startswith('_')]
print('\n'.join(methods))

In [ ]:
activity_id = 22545458432  # your 5x1K run

# 1. Lap/split data
laps = client.get_activity_splits(activity_id)
print("SPLITS keys:", laps.keys() if isinstance(laps, dict) else type(laps))

In [ ]:
laps = client.get_activity_splits(activity_id)

LAP_TO_INVESTIGATE = 0

print("Number of laps:", len(laps['lapDTOs']))
print("\nLap keys:", laps['lapDTOs'][LAP_TO_INVESTIGATE].keys())
print("\nLap data:")
print(json.dumps(laps['lapDTOs'][LAP_TO_INVESTIGATE], indent=2))

In [ ]:
# Extract just the date portion from the activity timestamp
activity_date = run_clean['date'].split('T')[0]  # "2026-04-16"
print("Activity date:", activity_date)

# 3. Training status / VO2 max

# Training status on day of activity
training_status = client.get_training_status(activity_date)
print("TRAINING STATUS:", json.dumps(training_status, indent=2))

# Also worth checking these on the same date while we're at it
hrv = client.get_hrv_data(activity_date)
print("HRV:", json.dumps(hrv, indent=2))

body_battery = client.get_body_battery(activity_date)
print("BODY BATTERY:", json.dumps(body_battery, indent=2))

In [ ]:
# HR Zone data
hr_zones = client.get_activity_hr_in_timezones(activity_id)
print(json.dumps(hr_zones, indent=2))

In [ ]:
# Sleep data
sleep = client.get_sleep_data(activity_date)
print("Top level keys:", sleep.keys())
print(json.dumps(sleep, indent=2))

#### Implement extractor for addtional features

##### Athlete Profile Extractor

In [ ]:
# Find out how to get bodyweight from the API
profile = client.get_user_profile()
print(json.dumps(profile, indent=2))

In [ ]:
def get_athlete_profile(client) -> dict:
    """Extract key athlete attributes from user profile."""
    profile = client.get_user_profile()
    data = profile['userData']
    
    return {
        'weight_kg': round((data.get('weight') or 0) / 1000, 1),
        'height_cm': data.get('height'),
        'gender': data.get('gender'),
        'vo2max_running': data.get('vo2MaxRunning'),
        'vo2max_cycling': data.get('vo2MaxCycling'),
        'lactate_threshold_hr': data.get('lactateThresholdHeartRate'),
        'lactate_threshold_pace': round(1 / ((data.get('lactateThresholdSpeed') or 1) * 0.06), 2),
    }

athlete = get_athlete_profile(client)
print(json.dumps(athlete, indent=2))

##### Splits Extractor

In [ ]:
def extract_laps_df(laps_data: dict) -> pd.DataFrame:
    """
    Convert Garmin lap DTOs to a DataFrame matching the Garmin Connect splits CSV format,
    with additional fields where available.
    """
    athlete = get_athlete_profile(client)
    body_weight_kg = athlete['weight_kg']

    rows = []
    interval_counter = 0
    cumulative_time = 0

    for lap in laps_data['lapDTOs']:
        intensity = lap.get('intensityType', '')
        distance = lap.get('distance') or 0
        duration = lap.get('duration') or 0
        moving_duration = lap.get('movingDuration') or 0
        avg_speed = lap.get('averageSpeed') or 0
        avg_moving_speed = lap.get('averageMovingSpeed') or 0
        gap_speed = lap.get('avgGradeAdjustedSpeed') or 0
        avg_power = lap.get('averagePower') or 0
        cumulative_time += duration

        # Format seconds to M:SS.s
        def fmt_time(secs):
            if not secs:
                return '--'
            m = int(secs // 60)
            s = secs % 60
            return f"{m}:{s:04.1f}"

        # Format speed (m/s) to pace (min/km) as M:SS string
        def fmt_pace(speed_ms):
            if not speed_ms or speed_ms <= 0:
                return '--'
            pace_secs = 1000 / speed_ms
            m = int(pace_secs // 60)
            s = int(pace_secs % 60)
            return f"{m}:{s:02d}"

        if intensity == 'ACTIVE':
            interval_counter += 1
            interval_label = str(interval_counter)
        else:
            interval_label = ''

        row = {
            'Interval':             interval_label,
            'Step Type':            intensity.replace('_', ' ').title() if intensity else '',
            'Lap':                  lap.get('lapIndex'),
            'Time':                 fmt_time(duration),
            'Cumulative Time':      fmt_time(cumulative_time),
            'Distance (km)':        round(distance / 1000, 2),
            'Avg Pace (min/km)':    fmt_pace(avg_speed),
            'Avg GAP (min/km)':     fmt_pace(gap_speed),
            'Moving Time':          fmt_time(moving_duration),
            'Avg Moving Pace':      fmt_pace(avg_moving_speed),
            'Avg HR':               lap.get('averageHR'),
            'Max HR':               lap.get('maxHR'),
            'Avg Cadence (spm)':    round(lap.get('averageRunCadence') or 0),
            'Max Cadence':          lap.get('maxRunCadence'),
            'Avg Power (W)':        avg_power,
            'NP (W)':               lap.get('normalizedPower'),
            'Max Power (W)':        lap.get('maxPower'),
            'Avg W/kg':             round(avg_power / body_weight_kg, 2) if avg_power else None,
            'GCT (ms)':             round(lap.get('groundContactTime') or 0, 1),
            'Stride Length (m)':    round((lap.get('strideLength') or 0) / 100, 2),
            'Vert Oscillation (cm)':round(lap.get('verticalOscillation') or 0, 1),
            'Vert Ratio (%)':       round(lap.get('verticalRatio') or 0, 1),
            'Elevation Gain (m)':   lap.get('elevationGain'),
            'Elevation Loss (m)':   lap.get('elevationLoss'),
            'Avg Respiration':      round(lap.get('avgRespirationRate') or 0, 1),
            'Avg Temp (°C)':        lap.get('averageTemperature'),
            'Calories':             lap.get('calories'),
            'Compliance Score':     lap.get('directWorkoutComplianceScore'),
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    return df


##### HR Zone Extractor

In [ ]:
def extract_hr_zones(hr_zones_data: list, total_duration_secs: float) -> list:
    """Convert HR zone data to a readable format with percentages."""
    zones = []
    for z in hr_zones_data:
        secs = z.get('secsInZone') or 0
        zones.append({
            'zone': z['zoneNumber'],
            'min_hr': z['zoneLowBoundary'],
            'time_min': round(secs / 60, 1),
            'pct_time': round(secs / total_duration_secs * 100, 1) if total_duration_secs else None,
        })
    return zones

##### Sleep Data

In [ ]:
def extract_sleep(sleep_data: dict) -> dict:
    """Extract key sleep metrics from Garmin sleep data."""
    dto = sleep_data['dailySleepDTO']
    scores = dto.get('sleepScores', {})
    sleep_need = dto.get('sleepNeed', {})

    total_secs = dto.get('sleepTimeSeconds') or 0

    def pct(secs):
        return round(secs / total_secs * 100, 1) if total_secs else None

    deep_secs  = dto.get('deepSleepSeconds') or 0
    light_secs = dto.get('lightSleepSeconds') or 0
    rem_secs   = dto.get('remSleepSeconds') or 0
    awake_secs = dto.get('awakeSleepSeconds') or 0

    return {
        'date':                 dto.get('calendarDate'),
        'sleep_score':          scores.get('overall', {}).get('value'),
        'sleep_score_label':    scores.get('overall', {}).get('qualifierKey'),
        'total_sleep_hrs':      round(total_secs / 3600, 2),
        'deep_sleep_hrs':       round(deep_secs / 3600, 2),
        'light_sleep_hrs':      round(light_secs / 3600, 2),
        'rem_sleep_hrs':        round(rem_secs / 3600, 2),
        'awake_hrs':            round(awake_secs / 3600, 2),
        'deep_pct':             pct(deep_secs),
        'light_pct':            pct(light_secs),
        'rem_pct':              pct(rem_secs),
        'awake_count':          dto.get('awakeCount'),
        'avg_hr':               dto.get('avgHeartRate'),
        'resting_hr':           sleep_data.get('restingHeartRate'),
        'avg_hrv':              sleep_data.get('avgOvernightHrv'),
        'hrv_status':           sleep_data.get('hrvStatus'),
        'avg_respiration':      dto.get('averageRespirationValue'),
        'avg_stress':           dto.get('avgSleepStress'),
        'body_battery_change':  sleep_data.get('bodyBatteryChange'),
        'sleep_need_hrs':       round((sleep_need.get('actual') or 0) / 60, 1),
        'sleep_need_feedback':  sleep_need.get('feedback'),
        'feedback':             dto.get('sleepScoreFeedback'),
    }

##### Testing the funcs

In [ ]:
laps_df = extract_laps_df(laps)
laps_df

In [ ]:
hr_zones_clean = extract_hr_zones(hr_zones, total_duration_secs=run_clean['duration_min'] * 60)

for z in hr_zones_clean:
    print(f"Zone {z['zone']} (>{z['min_hr']} bpm): {z['time_min']} min ({z['pct_time']}%)")


In [ ]:
sleep_clean = extract_sleep(sleep)
print(json.dumps(sleep_clean, indent=2))

## Summary

This notebook explored the `python-garminconnect` library as the foundation for a custom Garmin MCP server.

### Authentication
The library's authentication changed significantly in recent versions — `garth` is no longer used. 
Tokens are now stored in `garmin_tokens.json` via `client.client.dump()` / `client.client.load()`. 
Token persistence in a containerized environment will need further work during the server build phase.

### Endpoints Explored
The following API endpoints were validated and extraction functions written for each:

- **Activities** — `get_activities()` and `get_activity()` return rich sport-specific summaries including power, HR, cadence, and training load. Schema differs meaningfully across cycling, running, and swimming.
- **Lap splits** — `get_activity_splits()` returns per-lap data with full running dynamics (GCT, stride length, vertical oscillation, power). Structured into a DataFrame matching Garmin's splits CSV export format.
- **HR time-in-zones** — `get_activity_hr_in_timezones()` returns seconds and percentage per zone based on personal HR zone boundaries.
- **Sleep** — `get_sleep_data()` returns a comprehensive overnight summary including sleep score, stage breakdown, HRV, resting HR, body battery recovery, and sleep need.
- **HRV** — `get_hrv_data()` returns overnight average, weekly average, status, and personal baseline range. Raw 5-minute readings are available but not useful for MCP tools.
- **Body battery** — `get_body_battery()` returns daily charged/drained totals and per-event breakdown (sleep, activity, recovery).
- **Training status** — `get_training_status()` returns ACWR, acute/chronic load, training status label, load balance feedback, and sport-specific VO2max.
- **Athlete profile** — `get_user_profile()` returns weight (stored in grams), height, VO2max (running and cycling), lactate threshold HR, and lactate threshold pace.

### Extraction Functions Written
| Function | Purpose |
|---|---|
| `get_athlete_profile(client)` | Clean athlete attributes from user profile |
| `extract_activity_summary(activity)` | Sport-aware activity summary (cycling / running / swim) |
| `extract_laps_df(laps_data)` | Per-lap DataFrame with running dynamics and W/kg |
| `extract_hr_zones(hr_zones_data, total_duration_secs)` | HR zone time and percentage breakdown |
| `extract_sleep(sleep_data)` | Overnight sleep quality and recovery metrics |

### Planned MCP Tools
Based on the data available, the server will expose five tools:

| Tool | Description |
|---|---|
| `get_athlete_profile()` | Athlete attributes — weight, VO2max, LT HR, LT pace |
| `get_activities(limit, sport_type)` | Recent activity list with clean summaries |
| `get_activity(activity_id)` | Full activity detail with lap splits and HR zones |
| `get_sleep(date)` | Sleep quality and recovery summary for a given date |
| `get_daily_readiness(date)` | HRV, body battery, and training status for a given date |

### Next Steps
- Build `server.py` using FastMCP with SSE transport
- Implement token persistence strategy for containerized deployment
- Containerize and deploy to Azure Container Apps
- Connect to Claude.ai as a custom MCP connector

### Test for Coach Claude

In [ ]:
import json
from datetime import date

# ── CONFIG ──────────────────────────────────────────────────────────────────
# RUN_ACTIVITY_ID = 22545458432  # Ottawa - 5 x 1K @ 5K effort
RUN_ACTIVITY_ID = 22504555975    # Ottawa - 20K Long Run

# ── FETCH DATA ───────────────────────────────────────────────────────────────
print("Fetching data...")

activity_raw     = client.get_activity(RUN_ACTIVITY_ID)
laps_raw         = client.get_activity_splits(RUN_ACTIVITY_ID)
hr_zones_raw     = client.get_activity_hr_in_timezones(RUN_ACTIVITY_ID)
activity_date    = activity_raw['summaryDTO']['startTimeLocal'].split('T')[0]
sleep_raw        = client.get_sleep_data(activity_date)
hrv_raw          = client.get_hrv_data(activity_date)
body_battery_raw = client.get_body_battery(activity_date)
training_raw     = client.get_training_status(activity_date)
athlete          = get_athlete_profile(client)

print("All data fetched. Building report...")

# ── EXTRACT ───────────────────────────────────────────────────────────────────
activity  = extract_activity_summary(activity_raw)
laps_df   = extract_laps_df(laps_raw)
hr_zones  = extract_hr_zones(hr_zones_raw, activity['duration_min'] * 60)
sleep     = extract_sleep(sleep_raw)

# HRV summary
hrv = hrv_raw.get('hrvSummary', {})
hrv_clean = {
    'last_night_avg':  hrv.get('lastNightAvg'),
    'weekly_avg':      hrv.get('weeklyAvg'),
    'status':          hrv.get('status'),
    'baseline_low':    hrv.get('baseline', {}).get('balancedLow'),
    'baseline_high':   hrv.get('baseline', {}).get('balancedUpper'),
}

# Body battery
bb = body_battery_raw[0] if body_battery_raw else {}
bb_events = [
    {
        'type':     e.get('eventType'),
        'impact':   e.get('bodyBatteryImpact'),
        'feedback': e.get('shortFeedback'),
    }
    for e in bb.get('bodyBatteryActivityEvent', [])
]
bb_clean = {
    'charged':  bb.get('charged'),
    'drained':  bb.get('drained'),
    'events':   bb_events,
}

# Training status
ts = training_raw.get('mostRecentTrainingStatus', {}).get('latestTrainingStatusData', {})
device_id = list(ts.keys())[0] if ts else None
ts_data = ts.get(device_id, {}) if device_id else {}
acwr = ts_data.get('acuteTrainingLoadDTO', {})

load_balance = training_raw.get('mostRecentTrainingLoadBalance', {}) \
    .get('metricsTrainingLoadBalanceDTOMap', {})
lb_data = load_balance.get(device_id, {}) if device_id else {}

training_clean = {
    'status':           ts_data.get('trainingStatusFeedbackPhrase'),
    'sport':            ts_data.get('sport'),
    'acwr':             acwr.get('dailyAcuteChronicWorkloadRatio'),
    'acwr_status':      acwr.get('acwrStatus'),
    'load_balance':     lb_data.get('trainingBalanceFeedbackPhrase'),
}

vo2 = training_raw.get('mostRecentVO2Max', {})
vo2_clean = {
    'running':  vo2.get('generic', {}).get('vo2MaxPreciseValue'),
    'cycling':  vo2.get('cycling', {}).get('vo2MaxPreciseValue'),
}

# ── FORMAT REPORT ─────────────────────────────────────────────────────────────
lines = []
a = lines.append

a("=" * 60)
a("ATHLETE PROFILE")
a("=" * 60)
a(f"Weight:          {athlete['weight_kg']} kg")
a(f"VO2max (run):    {athlete['vo2max_running']} ml/kg/min")
a(f"VO2max (cycle):  {athlete['vo2max_cycling']} ml/kg/min")
a(f"LT Heart Rate:   {athlete['lactate_threshold_hr']} bpm")
a(f"LT Pace:         {athlete['lactate_threshold_pace']} min/km")

a("")
a("=" * 60)
a("RECOVERY — Night before activity")
a("=" * 60)
a(f"Date:               {sleep['date']}")
a(f"Sleep score:        {sleep['sleep_score']} ({sleep['sleep_score_label']})")
a(f"Total sleep:        {sleep['total_sleep_hrs']} hrs")
a(f"Deep / REM / Light: {sleep['deep_sleep_hrs']}h ({sleep['deep_pct']}%) / "
  f"{sleep['rem_sleep_hrs']}h ({sleep['rem_pct']}%) / "
  f"{sleep['light_sleep_hrs']}h ({sleep['light_pct']}%)")
a(f"Awake count:        {sleep['awake_count']}")
a(f"Avg HR:             {sleep['avg_hr']} bpm  |  Resting HR: {sleep['resting_hr']} bpm")
a(f"HRV (overnight):    {sleep['avg_hrv']} ms  ({sleep['hrv_status']})")
a(f"HRV baseline:       {hrv_clean['baseline_low']}–{hrv_clean['baseline_high']} ms  |  Weekly avg: {hrv_clean['weekly_avg']} ms")
a(f"Sleep stress:       {sleep['avg_stress']}")
a(f"Body battery:       +{sleep['body_battery_change']} overnight  |  Charged: {bb_clean['charged']}  Drained: {bb_clean['drained']}")
a(f"Sleep need:         {sleep['sleep_need_hrs']} hrs  ({sleep['sleep_need_feedback']})")
a(f"Feedback:           {sleep['feedback']}")

a("")
a("=" * 60)
a("TRAINING STATUS")
a("=" * 60)
a(f"Status:        {training_clean['status']}")
a(f"Sport:         {training_clean['sport']}")
a(f"ACWR:          {training_clean['acwr']}  ({training_clean['acwr_status']})")
a(f"Load balance:  {training_clean['load_balance']}")
a(f"VO2max run:    {vo2_clean['running']}  |  VO2max cycle: {vo2_clean['cycling']}")

a("")
a("=" * 60)
a(f"ACTIVITY — {activity['name']}")
a("=" * 60)
a(f"Date:              {activity['date']}")
a(f"Location:          {activity['location']}")
a(f"Distance:          {activity['distance_km']} km")
a(f"Duration:          {activity['duration_min']} min")
a(f"Elevation gain:    {activity['elevation_gain_m']} m")
a(f"Avg pace:          {activity['avg_pace_min_km']} min/km")
a(f"GAP:               {activity['grade_adjusted_pace']} min/km")
a(f"Avg HR / Max HR:   {activity['avg_hr']} / {activity['max_hr']} bpm")
a(f"Avg power / NP:    {activity['avg_power']}W / {activity['normalized_power']}W")
a(f"Avg cadence:       {activity['avg_cadence']} spm")
a(f"GCT:               {activity['ground_contact_ms']} ms")
a(f"Stride length:     {activity['stride_length_cm']} cm")
a(f"Vert oscillation:  {activity['vertical_oscillation_cm']} cm")
a(f"Vert ratio:        {activity['vertical_ratio']}%")
a(f"Training effect:   {activity['training_effect']} ({activity['training_effect_label']})")
a(f"Training load:     {activity['training_load']}")
a(f"Calories:          {activity['calories']}")
a(f"RPE:               {activity['rpe']} / 100")
a(f"Body battery Δ:    {activity['body_battery_change']}")

a("")
a("HR TIME IN ZONES")
a("-" * 40)
for z in hr_zones:
    bar = "█" * int(z['pct_time'] / 2)
    a(f"  Z{z['zone']} (>{z['min_hr']} bpm): {z['time_min']:5.1f} min  {z['pct_time']:5.1f}%  {bar}")

a("")
a("LAP SPLITS")
a("-" * 40)
print_cols = [
    'Lap', 'Step Type', 'Distance (km)', 'Avg Pace (min/km)',
    'Avg GAP (min/km)', 'Avg HR', 'Avg Power (W)', 'NP (W)',
    'Avg Cadence (spm)', 'GCT (ms)', 'Stride Length (m)',
    'Vert Oscillation (cm)', 'Compliance Score'
]
a(laps_df[print_cols].to_string(index=False))

report = "\n".join(lines)
print(report)